# Physical Neumann data and nonconvex macroelements in MH and MH²M

This notebook verifies the original boundary extensions with analytical data. MH retains the Robin multiplier $\lambda=(q-p\sigma)\cdot n$ and introduces pressure moments on Neumann faces. MH²M retains separate outward conormal coefficients $A\nabla p\cdot n$ and a continuous pressure trace. Neither multiplier is silently identified with the outward physical flux in the other formulation.

The cited methods are Barrenechea, Gomes and Paredes (2024), [DOI 10.1137/22M1542556](https://doi.org/10.1137/22M1542556), and de Barros, Madureira and Valentin, [MH²M version 3](https://arxiv.org/abs/2404.16978v3). Their homogeneous-Dirichlet numerical cases are distinct from this exact boundary verification.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from examples.mh_campaign import l_mesh
from pymhm.mesh import FaceSpace, SkeletonSpace
from pymhm.mh import solve_mh
from pymhm.mh2m import PressureTraceSpace, solve_mh2m

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pixi.toml").exists())


## Compatible pure Neumann patch

For $p=1+x^2+y^2$, $A=I$, $q=-2(x,y)$ and $f=-4$, prescribe $h_N=q\cdot n$ on the entire boundary. The compatibility identity is $\int_\Omega f=\int_{\partial\Omega}h_N=-4$. The volume mean is $5/3$. Two complementary L polygons cover the square; internal ear-triangulation diagonals remain local.

In [ ]:
mesh = l_mesh(1)
exact = lambda x: 1 + np.sum(x*x, axis=1)
physical_flux = lambda x: -2*x
neumann = {int(f): lambda x, n=mesh.normals[f]: physical_flux(x) @ n for f in mesh.boundary_faces}
common = dict(source=-4, dirichlet=None, neumann=neumann, mean_pressure=5/3, degree=2, local_refinement=4)
with threadpool_limits(1):
    mh = solve_mh(mesh, skeleton=SkeletonSpace(mesh, tuple(FaceSpace.uniform(2, 2) for _ in mesh.faces)), **common)
    mh2m = solve_mh2m(mesh, pressure_trace=PressureTraceSpace.uniform(mesh, 2),
                     flux_space=SkeletonSpace(mesh, tuple(FaceSpace.uniform(1, 2) for _ in mesh.faces)), **common)
checks = {}
for name, solution in (("MH", mh), ("MH2M", mh2m)):
    checks[name] = dict(pressure_L2=solution.l2_error(exact), flux_L2=solution.flux_l2_error(physical_flux),
                        macro_balance=float(np.max(abs(solution.conservation_residuals()))))
    assert max(checks[name].values()) < 2e-10
checks


## Five-resolution analytical campaign

The archived nonpolynomial case uses $A=\left[\begin{smallmatrix}3&0.4\\0.4&2\end{smallmatrix}\right]$ and $p=1+x+2y+\sin(\pi x)\sin(\pi y)$. All five series keep the same nonconvex geometry family and local P2 spaces. MH uses P2 Robin moments on two segments; MH²M uses continuous P2 pressure traces and P1 conormals on two segments. Mixed conditions put physical Neumann data on the right side and pressure elsewhere. Pure Neumann conditions prescribe the exact volume mean.

Error quadratures of orders eight and ten are compared independently of the assembly. Relative norms divide by the exact physical pressure or flux norm. The multilevel solve is separate from the notebook; its executed source hashes, original-equation diagnostics and dimensions are retained.

In [ ]:
record = json.loads((ROOT / "examples/results/mh/boundary-comparison.json").read_text())
assert not record["source_changed_during_run"]
assert len(record["rows"]) == 25
[{k: r[k] for k in ("method", "boundary", "resolution", "pressure_relative", "flux_relative")}
 for r in record["rows"] if r["resolution"] == max(x["resolution"] for x in record["rows"])]


In [ ]:
display(Image(filename=str(ROOT / "docs/figures/mh/boundary-convergence.png")))


The pressure/flux convergence and independent UFL block comparisons verify these boundary and geometry implementations. They do not establish the article's global Dirichlet SPD result for the augmented Neumann MH system, nor uniform stability for arbitrary independently chosen MH²M face spaces.